# 00 · Load check

Purpose: prove the environment works and take a first honest look at the data.

This notebook does **no** modeling. It answers four questions:

1. Does the file load, and is it the shape we expect?
2. How rare is fraud, really?
3. Is anything missing?
4. What scale are the raw features on?

Every number below has an expected value. If one doesn't match, stop and fix it before Phase 1.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

df = pd.read_csv("../data/raw/creditcard.csv")
df.shape

### Check 1 — shape

Expected: **(284807, 31)** — 284,807 transactions, 31 columns.

The 31 columns are `Time`, `Amount`, `Class`, and `V1`–`V28`.

In [ ]:
assert df.shape == (284807, 31), f"unexpected shape: {df.shape}"
print("columns:", list(df.columns))

### Check 2 — class balance

This is the single most important number in the project.

Expected: **492 fraud out of 284,807 = 0.1727%**.

In [ ]:
counts = df["Class"].value_counts()
rates  = df["Class"].value_counts(normalize=True)

print(f"legitimate : {counts[0]:>7,}   ({rates[0]:.4%})")
print(f"fraud      : {counts[1]:>7,}   ({rates[1]:.4%})")
print(f"ratio      : 1 fraud per {counts[0] / counts[1]:.0f} legitimate transactions")

**Why this matters — the trap this project is built around.**

A model that predicts "never fraud" for every transaction achieves:

- accuracy: **99.83%**
- fraud caught: **0**

Accuracy is not just a weak metric here, it is actively misleading. Nothing in this repo reports it.
Phase 1 uses precision, recall, F1 and PR-AUC instead.

Sanity-check that claim rather than taking it on faith:

In [ ]:
naive_predictions = np.zeros(len(df), dtype=int)   # "never fraud"
accuracy = (naive_predictions == df["Class"]).mean()
recall   = naive_predictions[df["Class"] == 1].sum() / (df["Class"] == 1).sum()

print(f"naive model accuracy : {accuracy:.4%}")
print(f"naive model recall   : {recall:.4%}   <- catches nothing")

### Check 3 — missing values

Expected: **0**. This dataset is already clean, which is unusual and is one reason it's a good
teaching set — we get to spend our time on the imbalance problem instead of on data cleaning.

In [ ]:
missing = df.isnull().sum().sum()
print("total missing values:", missing)
assert missing == 0

### Check 4 — feature scales

`V1`–`V28` are PCA components, so they are already centred near zero with comparable spread.
`Time` and `Amount` are **not** — they're raw.

Look at the difference in `std` and `max` below. That gap is exactly why Phase 1 scales those two
columns: distance- and reconstruction-based models (Isolation Forest, autoencoder) would otherwise
let `Amount` and `Time` dominate the geometry purely because their numbers are bigger.

In [ ]:
df[["Time", "Amount"]].describe()

In [ ]:
# for contrast, a few of the PCA components
df[["V1", "V2", "V3", "V4"]].describe()

### What `Time` actually is

Seconds elapsed since the first transaction in the dataset — **not** a timestamp. The whole file
spans two days.

This matters for Phase 1: a random train/test split would let the model train on later transactions
and test on earlier ones, which leaks information a real deployment would never have. A time-based
hold-out (train on day 1, test on day 2) mimics reality more honestly.

In [ ]:
span_seconds = df["Time"].max()
print(f"span: {span_seconds:,.0f} seconds = {span_seconds/3600:.1f} hours = {span_seconds/86400:.2f} days")
print(f"fraud in first half : {df[df.Time <= span_seconds/2].Class.sum()}")
print(f"fraud in second half: {df[df.Time >  span_seconds/2].Class.sum()}")

### Amount, split by class

Worth a look before Phase 1's EDA: is fraud systematically larger or smaller?

In [ ]:
df.groupby("Class")["Amount"].describe()[["count", "mean", "50%", "max"]]

---

## Result

If every assertion above passed, Phase 0 is done:

- environment installs and imports cleanly
- dataset loads at the expected shape
- the 0.172% fraud rate is confirmed first-hand

**Next:** Phase 1 — EDA, preprocessing, and three models compared on metrics that survive this
level of class imbalance.